# Word vectors for the full wiki corpus (GPU)

Runs `wikiembed parse` (Stanza) and `wikiembed embed` (XLM-R) over **all topics, every sentence** of the collected Wikipedia data.

**Before running**
1. On your computer: `python scripts/make_colab_bundle.py`, then upload `outputs/colab_bundle.zip` to Google Drive as **`MyDrive/nlp/colab_bundle.zip`**.
2. Here: **Runtime → Change runtime type → T4 GPU**.
3. **Runtime → Run all**, and allow Drive access when asked.

Results are written straight to `MyDrive/nlp/embed_all/`. If the session disconnects, just **Run all** again: parsing resumes where it stopped.

In [ ]:
!nvidia-smi -L || echo 'No GPU: Runtime > Change runtime type > T4 GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BUNDLE = '/content/drive/MyDrive/nlp/colab_bundle.zip'
OUT = '/content/drive/MyDrive/nlp/embed_all'   # results live on Drive, so a disconnect loses nothing

In [ ]:
!rm -rf /content/nlp && mkdir -p /content/nlp && unzip -q "{BUNDLE}" -d /content/nlp
%cd /content/nlp
!pip install -q stanza pysbd
!ls -lh data/wiki

In [ ]:
import stanza
for lang in ['en', 'es', 'ja']:
    stanza.download(lang, verbose=False)
print('Stanza models ready')

In [ ]:
# Step 1: Stanza words / lemmas / POS for every sentence (resumable)
!python -m wikiembed parse --all-topics --max-per-article 0 --wiki-dir data/wiki --out-dir "{OUT}"

In [ ]:
# Step 2: XLM-R type-level vectors (one per lang, lemma, POS), mean-centered per language
!python -m wikiembed embed --all-topics --max-per-article 0 --out-dir "{OUT}" --batch-size 128

In [ ]:
# Sanity check: translations should be the top cross-language neighbours
!python -m wikiembed neighbors 光合成 --lang ja --target en --out-dir "{OUT}" -k 5
!python -m wikiembed neighbors energía --lang es --target ja --out-dir "{OUT}" -k 5
!ls -lh "{OUT}"

**Done.** Download from `MyDrive/nlp/embed_all/` into `data/embed_all/` on your computer:
- `vectors.npz`, `vocab.jsonl`, `embed_config.json`: the word vectors (needed for reconstruction)
- `parsed/`: Stanza output for every sentence (large; only needed for tree and discourse measures)

Load with `WordVectors.load("data/embed_all")` from `wikiembed.vectors`.